# Project 10: Overfitting and Underfitting Analysis

**Goal:** Train neural networks of different sizes on the Iris dataset, use the *same* train/test split for all of them, and see what happens when a model is too simple, about right, or too complex.

**Plan**
1. Load and prepare the data
2. Build three models: small, medium, large
3. Compare training vs. testing accuracy
4. Look at how complexity and training time change the picture
5. Try fixes for overfitting
6. Conclusions

## Step 1: Load the data

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore")   # keeps the output clean (sklearn warns when training stops early)
iris = load_iris()
print("Full data shape:", iris.data.shape)
print("Features:", iris.feature_names)
print("Classes:", list(iris.target_names))

### A note on which features I use

With all 4 Iris features, even a basic model gets ~95%+ accuracy, so the gap between training and testing accuracy is tiny and overfitting is hard to see. 
To make the problem a bit harder (and the effect easier to observe), I use only the **two sepal features**. 
Versicolor and virginica overlap a lot in these two features, so a model that tries to be perfect on the training data has to memorise noise.

In [ ]:
X = iris.data[:, :2]      # sepal length, sepal width
y = iris.target

plt.figure(figsize=(6, 4))
for i, name in enumerate(iris.target_names):
    plt.scatter(X[y == i, 0], X[y == i, 1], label=name, alpha=0.7)
plt.xlabel("sepal length (cm)")
plt.ylabel("sepal width (cm)")
plt.title("Iris: sepal features only")
plt.legend()
plt.show()

### Train/test split

Every model below uses this exact same split. I scale the data using only the training set so nothing from the test set leaks in.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=0, stratify=y
)

scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

print("Train size:", len(X_train), "| Test size:", len(X_test))

## Step 2: Build three models

| Model | Architecture | Idea |
|---|---|---|
| Simple | 1 hidden neuron | Too small to learn the pattern |
| Medium | 1 hidden layer, 6 neurons, light regularization | Should be about right |
| Complex | 3 hidden layers x 256 neurons, no regularization, trained for a long time | Enough capacity to memorise the training set |

In [ ]:
def make_models():
    return {
        "Simple (1 neuron)": MLPClassifier(
            hidden_layer_sizes=(1,), max_iter=1000,
            learning_rate_init=0.01, random_state=0),

        "Medium (6 neurons)": MLPClassifier(
            hidden_layer_sizes=(6,), alpha=0.1, max_iter=1000,
            learning_rate_init=0.01, random_state=0),

        "Complex (3x256)": MLPClassifier(
            hidden_layer_sizes=(256, 256, 256), alpha=0, max_iter=2000,
            tol=1e-8, n_iter_no_change=2000,     # don't let it stop early
            learning_rate_init=0.01, random_state=0),
    }

models = make_models()
for name, model in models.items():
    model.fit(X_train, y_train)
    print("Trained:", name)

## Step 3: Compare training and testing performance

In [ ]:
rows = []
for name, model in models.items():
    train_acc = model.score(X_train, y_train)
    test_acc = model.score(X_test, y_test)
    rows.append({"Model": name,
                 "Train accuracy": round(train_acc, 3),
                 "Test accuracy": round(test_acc, 3),
                 "Gap (train - test)": round(train_acc - test_acc, 3)})

results = pd.DataFrame(rows)
results

In [ ]:
x_pos = np.arange(len(results))
width = 0.35

plt.figure(figsize=(8, 4))
plt.bar(x_pos - width/2, results["Train accuracy"], width, label="Train")
plt.bar(x_pos + width/2, results["Test accuracy"], width, label="Test")
plt.xticks(x_pos, results["Model"])
plt.ylim(0, 1.05)
plt.ylabel("Accuracy")
plt.title("Train vs. test accuracy")
plt.legend()
plt.show()

### How to read this

- **Underfitting:** both train and test accuracy are low. The model is too simple to capture the pattern.
- **Good fit:** train and test accuracy are both decent and close to each other.
- **Overfitting:** train accuracy is very high but test accuracy is clearly lower. The model memorised the training data, including its noise.

The table above should show all three of these. (Exact numbers depend on the split, so I check this on more splits in Step 6.)

### Decision boundaries

Plotting what each model actually learned makes the difference obvious. The complex model draws a messy boundary that wraps around individual training points.

In [ ]:
xx, yy = np.meshgrid(np.linspace(X_train[:, 0].min() - 0.5, X_train[:, 0].max() + 0.5, 300),
                     np.linspace(X_train[:, 1].min() - 0.5, X_train[:, 1].max() + 0.5, 300))
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, model) in zip(axes, models.items()):
    zz = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.25, cmap="viridis")
    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="viridis", edgecolor="k", s=30)
    ax.set_title(f"{name}\ntest acc = {model.score(X_test, y_test):.2f}")
    ax.set_xlabel("sepal length (scaled)")
    ax.set_ylabel("sepal width (scaled)")
plt.tight_layout()
plt.show()

## Step 4: How does complexity change things?

Next I train networks with a growing number of neurons in one hidden layer (everything else held constant) and plot train vs. test accuracy. 
Testing performance should rise, then flatten or drop while the training accuracy keeps climbing.

In [ ]:
sizes = [1, 2, 4, 8, 16, 64, 256]
train_scores, test_scores = [], []

for n in sizes:
    m = MLPClassifier(hidden_layer_sizes=(n,), alpha=0, max_iter=2000,
                      tol=1e-8, n_iter_no_change=2000,
                      learning_rate_init=0.01, random_state=0)
    m.fit(X_train, y_train)
    train_scores.append(m.score(X_train, y_train))
    test_scores.append(m.score(X_test, y_test))

plt.figure(figsize=(7, 4))
plt.plot(sizes, train_scores, "o-", label="Train")
plt.plot(sizes, test_scores, "o-", label="Test")
plt.xscale("log", base=2)
plt.xlabel("Neurons in hidden layer (log scale)")
plt.ylabel("Accuracy")
plt.title("Model complexity vs. accuracy")
plt.legend()
plt.show()

### Training time matters too

A big model also overfits more the longer it trains. Here I train the complex model one epoch at a time and track accuracy on both sets.

In [ ]:
big = MLPClassifier(hidden_layer_sizes=(256, 256, 256), alpha=0,
                    learning_rate_init=0.01, random_state=0)

epochs = 600
train_hist, test_hist = [], []
classes = np.unique(y_train)

for epoch in range(epochs):
    big.partial_fit(X_train, y_train, classes=classes)
    train_hist.append(big.score(X_train, y_train))
    test_hist.append(big.score(X_test, y_test))

plt.figure(figsize=(7, 4))
plt.plot(train_hist, label="Train")
plt.plot(test_hist, label="Test")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Complex model: accuracy over training time")
plt.legend()
plt.show()

## Step 5: Fixing overfitting

Ways to reduce overfitting:

1. **Make the model smaller:** fewer neurons or layers.
2. **Regularization (L2):** penalise large weights (`alpha` in sklearn).
3. **Early stopping:** stop training when the validation score stops improving.
4. **More data:** harder to memorise a bigger dataset (not an option with Iris, so I skip it).

I apply the first three to the complex model and compare against the original.

In [ ]:
fixes = {
    "Original complex": MLPClassifier(
        hidden_layer_sizes=(256, 256, 256), alpha=0, max_iter=2000,
        tol=1e-8, n_iter_no_change=2000, learning_rate_init=0.01, random_state=0),

    "Smaller network (6)": MLPClassifier(
        hidden_layer_sizes=(6,), alpha=0, max_iter=2000,
        tol=1e-8, n_iter_no_change=2000, learning_rate_init=0.01, random_state=0),

    "L2 regularization": MLPClassifier(
        hidden_layer_sizes=(256, 256, 256), alpha=1.0, max_iter=2000,
        tol=1e-8, n_iter_no_change=2000, learning_rate_init=0.01, random_state=0),

    "Early stopping": MLPClassifier(
        hidden_layer_sizes=(256, 256, 256), alpha=0, max_iter=2000,
        early_stopping=True, validation_fraction=0.2, n_iter_no_change=10,
        learning_rate_init=0.01, random_state=0),
}

rows = []
for name, m in fixes.items():
    m.fit(X_train, y_train)
    tr, te = m.score(X_train, y_train), m.score(X_test, y_test)
    rows.append({"Model": name, "Train accuracy": round(tr, 3),
                 "Test accuracy": round(te, 3), "Gap": round(tr - te, 3),
                 "Epochs run": m.n_iter_})

pd.DataFrame(rows)

## Step 6: Is this result reliable?

Iris is small (150 rows, only 60 in the test set here), so a single split can be misleading. One lucky or unlucky split changes accuracy by several percent. 
To check, I repeat the experiment on 10 different random splits and average.

In [ ]:
def run_once(seed):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4, random_state=seed, stratify=y)
    sc = StandardScaler().fit(Xtr)
    Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)
    out = {}
    for name, m in make_models().items():
        m.fit(Xtr, ytr)
        out[name] = (m.score(Xtr, ytr), m.score(Xte, yte))
    return out

all_runs = [run_once(s) for s in range(10)]

summary = []
for name in all_runs[0]:
    tr = np.array([r[name][0] for r in all_runs])
    te = np.array([r[name][1] for r in all_runs])
    summary.append({"Model": name,
                    "Train (mean)": round(tr.mean(), 3),
                    "Test (mean)": round(te.mean(), 3),
                    "Test (std)": round(te.std(), 3),
                    "Gap": round(tr.mean() - te.mean(), 3)})
pd.DataFrame(summary)

## Conclusions

**Underfitting (1 neuron):** Training and testing accuracy are both low. The model doesn't have enough capacity to separate the classes, so adding more training time wouldn't help. It needs a bigger model.

**Good fit (6 neurons + light regularization):** Train and test accuracy are both reasonable and close to each other. This model generalises best, and has the highest test accuracy of the three.

**Overfitting (3x256, no regularization):** Training accuracy is close to 100%, but test accuracy is lower than the medium model. The decision boundary also gets jagged. It memorised the training points, noise included.

**What helped:** a smaller network, L2 regularization, and early stopping all reduced the train/test gap (see Step 5).

**Limitations**
- Iris is tiny, so results move around from split to split. Step 6 shows the averages are more trustworthy than one run.
- I used only two features on purpose to make the effect visible. With all four features, Iris is too easy and the overfitting effect mostly disappears.
- Test accuracy was used to compare models. In a real project I'd keep a separate validation set for choosing the model and only touch the test set at the end.